# EDA with SQL — Falcon 9 Launch Data

This notebook demonstrates the SQL analyses in the IBM capstone using an in-memory SQLite copy of the frozen 90-record course modelling dataset. The SQL statements use standard SQL syntax. This is a reproducible local companion to the IBM Db2 lab, not a claim that these queries were run on the learner's Db2 service.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

DATA=Path('data/dataset_part_2.csv')
if not DATA.exists(): raise FileNotFoundError(DATA)
df=pd.read_csv(DATA)
df.columns=[str(c).strip() for c in df.columns]
required={'LaunchSite','PayloadMass','Orbit','Class','Date','BoosterVersion','Outcome'}
missing=required-set(df.columns)
if missing: raise ValueError(f'Missing SQL columns: {sorted(missing)}')
df['PayloadMass']=pd.to_numeric(df['PayloadMass'],errors='coerce')
df['Class']=pd.to_numeric(df['Class'],errors='coerce')
df['Date']=pd.to_datetime(df['Date'],errors='coerce').dt.strftime('%Y-%m-%d')
con=sqlite3.connect(':memory:')
df.to_sql('SPACEXTBL',con,index=False,if_exists='replace')
print('Loaded rows into SPACEXTBL:',len(df))

def query(sql):
    result=pd.read_sql_query(sql,con)
    display(result)
    return result

Loaded rows into SPACEXTBL: 90


## 1. Identify unique launch sites

In [2]:
q1=query('SELECT DISTINCT LaunchSite FROM SPACEXTBL ORDER BY LaunchSite;')

,LaunchSite
0,CCAFS SLC 40
1,KSC LC 39A
2,VAFB SLC 4E


## 2. Count launches by site and calculate observed landing success rate

In [3]:
q2=query('''SELECT LaunchSite, COUNT(*) AS Launches,
       ROUND(100.0 * AVG(Class), 1) AS SuccessRatePct
FROM SPACEXTBL GROUP BY LaunchSite ORDER BY Launches DESC;''')

,LaunchSite,Launches,SuccessRatePct
0,CCAFS SLC 40,55,60.0
1,KSC LC 39A,22,77.3
2,VAFB SLC 4E,13,76.9


## 3. Calculate total and average payload by orbit

In [4]:
q3=query('''SELECT Orbit, COUNT(PayloadMass) AS PayloadRecords,
       ROUND(SUM(PayloadMass), 1) AS TotalPayloadKg,
       ROUND(AVG(PayloadMass), 1) AS AveragePayloadKg
FROM SPACEXTBL GROUP BY Orbit ORDER BY TotalPayloadKg DESC;''')

,Orbit,PayloadRecords,TotalPayloadKg,AveragePayloadKg
0,VLEO,14,214420.0,15315.7
1,GTO,27,135323.9,5012.0
2,ISS,21,68878.7,3279.9
3,PO,9,68253.0,7583.7
4,LEO,7,27179.9,3882.8
5,MEO,3,11961.0,3987.0
6,SSO,5,10300.0,2060.0
7,SO,1,6105.0,6105.0
8,GEO,1,6105.0,6105.0
9,ES-L1,1,570.0,570.0


## 4. Calculate average payload by booster version

In [5]:
q4=query('''SELECT BoosterVersion, COUNT(*) AS Launches,
       ROUND(AVG(PayloadMass),1) AS AveragePayloadKg
FROM SPACEXTBL GROUP BY BoosterVersion ORDER BY AveragePayloadKg DESC;''')

,BoosterVersion,Launches,AveragePayloadKg
0,Falcon 9,90,6105.0


## 5. Identify the earliest successful landing record

This query uses the binary Class field and date. It is distinct from the overall mission-success measure.

In [6]:
q5=query('''SELECT MIN(Date) AS EarliestSuccessfulLandingDate
FROM SPACEXTBL WHERE Class=1;''')

,EarliestSuccessfulLandingDate
0,2014-04-18


## 6. Count landing outcomes by outcome description

In [7]:
q6=query('''SELECT Outcome, COUNT(*) AS Records
FROM SPACEXTBL GROUP BY Outcome ORDER BY Records DESC;''')

,Outcome,Records
0,True ASDS,41
1,None None,19
2,True RTLS,14
3,False ASDS,6
4,True Ocean,5
5,None ASDS,2
6,False Ocean,2
7,False RTLS,1


## 7. Find booster versions associated with the maximum payload mass

In [8]:
q7=query('''SELECT DISTINCT BoosterVersion, PayloadMass
FROM SPACEXTBL
WHERE PayloadMass=(SELECT MAX(PayloadMass) FROM SPACEXTBL);''')

,BoosterVersion,PayloadMass
0,Falcon 9,15600.0


## 8. Find unsuccessful landing attempts in 2015 by outcome category

In [9]:
q8=query('''SELECT BoosterVersion, LaunchSite, Date, Outcome
FROM SPACEXTBL
WHERE Class=0 AND Date >= '2015-01-01' AND Date < '2016-01-01'
ORDER BY Date;''')

,BoosterVersion,LaunchSite,Date,Outcome
0,Falcon 9,CCAFS SLC 40,2015-01-10,False ASDS
1,Falcon 9,CCAFS SLC 40,2015-04-14,False ASDS
2,Falcon 9,CCAFS SLC 40,2015-04-27,None None
3,Falcon 9,CCAFS SLC 40,2015-06-28,None ASDS


## 9. Rank sites by successful landing count

In [10]:
q9=query('''SELECT LaunchSite, SUM(CASE WHEN Class=1 THEN 1 ELSE 0 END) AS SuccessfulLandings
FROM SPACEXTBL GROUP BY LaunchSite ORDER BY SuccessfulLandings DESC;''')

,LaunchSite,SuccessfulLandings
0,CCAFS SLC 40,33
1,KSC LC 39A,17
2,VAFB SLC 4E,10


## 10. Analyze yearly launch volume and landing success

In [11]:
q10=query('''SELECT SUBSTR(Date,1,4) AS LaunchYear,
       COUNT(*) AS Launches,
       ROUND(100.0*AVG(Class),1) AS SuccessRatePct
FROM SPACEXTBL WHERE Date IS NOT NULL
GROUP BY SUBSTR(Date,1,4) ORDER BY LaunchYear;''')

,LaunchYear,Launches,SuccessRatePct
0,2010,1,0.0
1,2012,1,0.0
2,2013,3,0.0
3,2014,6,33.3
4,2015,6,33.3
5,2016,8,62.5
6,2017,18,83.3
7,2018,18,61.1
8,2019,10,90.0
9,2020,19,84.2


## Interpretation and scope

Use the returned query tables in the presentation and explain each result in the context of the 90-row modelling dataset. Do not substitute numbers from the separate 101-row Db2 lab dataset; their populations and filters differ.